# 1. Limpieza de datos y separación train/test

**Materia:** 72.75 Aprendizaje Automático · ITBA · 2026 Q2 · TP3: Competencia de ML  
**Integrantes:** Diego Emiliano Godinez Ortega, Thomas Michael Spitzner, Guillermo Arturo Ledesma Campos

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

import sys
from pathlib import Path

PROJECT_PATH = Path("..").resolve()
DATA_PATH = PROJECT_PATH / "data"
FIGURES_PATH = PROJECT_PATH / "figures"
sys.path.append(str(PROJECT_PATH))
RANDOM_STATE = 42

from predictor import TARGET, LEAKAGE_COLUMNS

## 1.1 Carga del dataset

In [2]:
raw_reservations = pd.read_csv(DATA_PATH / "gpu_resource_reservations.csv")
print(f"{raw_reservations.shape[0]:,} reservas × {raw_reservations.shape[1]} columnas")
raw_reservations.head()

47,842 reservas × 34 columnas


,canceled_job,hours_before_execution,execution_year,execution_month,execution_week,execution_day,peak_gpu_hours,offpeak_gpu_hours,num_primary_processes,num_auxiliary_processes,...,queue_wait_time,workload_profile,estimated_cost_per_hour,reserved_storage_units,special_resource_requests,execution_status,execution_status_date,total_processes,total_gpu_hours,high_value_client
0,0,6,2015,7,27,1,0,16,1,0.0,...,0,single_experiment,0.000,0,0,completed,2015-07-03,1.0,16,0
1,1,88,2015,7,27,1,0,32,2,0.0,...,0,single_experiment,3.825,0,1,canceled,2015-07-01,2.0,32,0
2,1,92,2015,7,27,1,16,32,2,0.0,...,0,single_experiment,3.825,0,2,canceled,2015-06-23,2.0,48,0
3,1,79,2015,7,27,2,0,24,2,0.0,...,0,single_experiment,3.825,0,1,canceled,2015-06-25,2.0,24,0
4,1,62,2015,7,27,2,16,24,2,0.0,...,0,single_experiment,3.825,0,1,no_show,2015-07-02,2.0,40,0


## 1.2 Data leakage: `execution_status` y `execution_status_date`

In [3]:
pd.crosstab(raw_reservations["execution_status"], raw_reservations[TARGET])

canceled_job,0,1
execution_status,,
canceled,0,19416
completed,27887,0
no_show,0,539


- `execution_status` reproduce el target (`canceled` / `no_show` → 1) y `execution_status_date` se conoce después del resultado.
- Se eliminan: no llegan al predictor en la evaluación.

In [4]:
reservations = raw_reservations.drop(columns=LEAKAGE_COLUMNS)
reservations.shape

(47842, 32)

## 1.3 Valores faltantes

In [5]:
missing_values = reservations.isna().sum()
missing_values = missing_values[missing_values > 0]
pd.DataFrame({"missing": missing_values, "percentage": (missing_values / len(reservations) * 100).round(2)})

,missing,percentage
num_auxiliary_processes,1,0.00
client_region,20,0.04
orchestration_agent_id,5035,10.52
organization_id,45540,95.19


- `orchestration_agent_id` y `organization_id`: faltante = reserva sin agente / sin organización → categoría `missing`.
- `client_region` y `num_auxiliary_processes`: casos aislados → `missing` y 0.
- La imputación está en `engineer_features` (`predictor.py`) para aplicarse igual en train, test y evaluación.

## 1.4 Consistencia de variables derivadas

In [6]:
expected_total_processes = (reservations["num_primary_processes"]
                            + reservations["num_auxiliary_processes"].fillna(0)
                            + reservations["num_background_processes"])
expected_total_gpu_hours = reservations["peak_gpu_hours"] + reservations["offpeak_gpu_hours"]
print("total_processes inconsistentes:", (reservations["total_processes"] != expected_total_processes).sum())
print("total_gpu_hours inconsistentes:", (reservations["total_gpu_hours"] != expected_total_gpu_hours).sum())

total_processes inconsistentes: 0
total_gpu_hours inconsistentes: 0


## 1.5 Valores fuera de dominio

In [7]:
domain_checks = pd.Series({
    "total_processes = 0": (reservations["total_processes"] == 0).sum(),
    "total_gpu_hours = 0": (reservations["total_gpu_hours"] == 0).sum(),
    "estimated_cost_per_hour = 0": (reservations["estimated_cost_per_hour"] == 0).sum(),
    "estimated_cost_per_hour < 0": (reservations["estimated_cost_per_hour"] < 0).sum(),
})
print(domain_checks)
print("\nMayores costos por hora:", reservations["estimated_cost_per_hour"].nlargest(5).tolist())

total_processes = 0            108
total_gpu_hours = 0            193
estimated_cost_per_hour = 0    727
estimated_cost_per_hour < 0      0
dtype: int64

Mayores costos por hora: [270.0, 25.5, 18.6165, 17.625, 17.4815]


- Casos raros pero posibles (reservas sin procesos o sin costo, un costo extremo de 270): se mantienen.
- El predictor tiene que devolver una predicción para todas las reservas activas, también estas.

## 1.6 Duplicados

In [8]:
duplicated_mask = reservations.duplicated(keep=False)
print(f"Filas idénticas a otra: {duplicated_mask.sum():,} ({duplicated_mask.mean():.1%})")
print(f"Copias redundantes: {reservations.duplicated().sum():,}")
reservations.groupby(duplicated_mask)[TARGET].mean().rename(index={False: "única", True: "duplicada"}).rename("tasa de cancelación")

Filas idénticas a otra: 19,347 (40.4%)
Copias redundantes: 15,857


única        0.284857
duplicada    0.611878
Name: tasa de cancelación, dtype: float64

- Son reservas distintas con los mismos datos (reservas grupales), no errores de carga: se mantienen.
- La evaluación es aleatoria sobre la misma población, así que el lote de evaluación también las contiene.

## 1.7 Separación train/test 80/20 estratificada
Aleatoria, igual que el conjunto de evaluación. Todo el análisis siguiente usa solo train.

In [9]:
train_reservations, test_reservations = train_test_split(
    reservations, test_size=0.20, stratify=reservations[TARGET], random_state=RANDOM_STATE
)
train_reservations.to_csv(DATA_PATH / "train.csv", index=False)
test_reservations.to_csv(DATA_PATH / "test.csv", index=False)

pd.DataFrame({
    "reservas": [len(train_reservations), len(test_reservations)],
    "tasa de cancelación": [train_reservations[TARGET].mean(), test_reservations[TARGET].mean()],
}, index=["train", "test"]).round(4)

,reservas,tasa de cancelación
train,38273,0.4171
test,9569,0.4171
